# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MONTH = "2026-03"
FACT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
T = "2026-03-15"

data = con.execute(f"""
    WITH feat AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions)      AS avg_impressions,
               AVG(gsc_clicks)           AS avg_clicks,
               AVG(gsc_avg_position)     AS avg_position,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS ctr,
               COUNT(DISTINCT report_date) AS days_seen
        FROM '{FACT}'
        WHERE report_date < DATE '{T}'
        GROUP BY 1, 2
    ),
    tgt AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions) AS tgt_avg_impressions
        FROM '{FACT}'
        WHERE report_date >= DATE '{T}'
        GROUP BY 1, 2
    )
    SELECT f.*, t.tgt_avg_impressions,
           CASE WHEN f.avg_impressions > 5
                 AND t.tgt_avg_impressions < 0.8 * f.avg_impressions
                THEN 1 ELSE 0 END AS label_declined
    FROM feat f
    JOIN tgt t USING (content_hash_id, client_hash_id)
""").df().dropna(subset=['avg_impressions', 'avg_clicks', 'avg_position', 'ctr', 'days_seen'])

print("Shape:", data.shape)
print("Base rate (label_declined == 1):", round(data['label_declined'].mean(), 4))
print("Distinct clients:", data['client_hash_id'].nunique())
data.head()

HF_TOKEN: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (150443, 9)
Base rate (label_declined == 1): 0.1857
Distinct clients: 44


,content_hash_id,client_hash_id,avg_impressions,avg_clicks,avg_position,ctr,days_seen,tgt_avg_impressions,label_declined
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,6.714286,0.000000,5.285458,0.000000,14,5.117647,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,1.428571,0.000000,3.597222,0.000000,14,0.823529,0
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,4.000000,0.000000,3.635374,0.000000,14,1.235294,0
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,16.928571,0.000000,4.542255,0.000000,14,5.411765,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,12.785714,0.142857,4.105758,0.011173,14,24.882353,0


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**My question shape:** "which pages should be reviewed first?" — a yes/no observed label (`label_declined`) used for ranking, not just classifying. Per the training-honest-models skill's table, that shape says: start with **Logistic Regression** (readable), then try **Random Forest** (stronger) — add complexity only if it earns its keep on the comparison table in Section 3.

**Why Logistic Regression first:** with only 5 features and one month of data, a linear model is easy to read — I can look at its coefficients and immediately sanity-check whether the direction of each one makes sense (e.g. does low CTR push the decline probability up?). It's also a fair, simple thing to beat my rule-based baseline with; if it can't beat a simple `if`-`then` rule, a more complex model probably can't either.

**Why Random Forest second:** it can capture interactions between features that a linear model can't (e.g. "low CTR only matters when position is also good") without me hand-writing every interaction term. I use it as a stronger candidate, not a default choice — the comparison table decides if the extra complexity was worth it.

**Why not gradient boosting this week:** with one month, ~5 features, and a single train/test split (not cross-validated), gradient boosting is more prone to overfitting and harder to interpret than the two methods above are. I'm treating it as a stretch goal, not this week's model.

**Ranking mechanics:** both models output a predicted *probability* of decline for each page. I use that probability directly as the ranking score (highest probability = reviewed first) and evaluate with precision@K, exactly like the baseline's `baseline_action_score` — same use case, same yardstick.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped-by-client split — not a plain random split.** Pages from the same client tend to share patterns (the same site structure, the same content strategy, similar traffic behavior). If I split randomly, some of a client's pages could end up in training and others in testing — the model could partly "memorize" that client's quirks rather than learn signals that generalize, and my test score would look better than it deserves. `client_hash_id` is used only to build the split; it is never a model feature.

**Not a time-aware split this week.** My whole dataset lives inside one month with one fixed cutoff T = 2026-03-15 — there's no second time period to hold out as a genuinely later test window yet. A per-client time-aware split becomes possible once the capstone moves to multiple months; for this week, client-grouping is the honest split available to me.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

FEATURES = ['avg_impressions', 'avg_clicks', 'avg_position', 'ctr', 'days_seen']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))

train_df = data.iloc[train_idx].reset_index(drop=True)
test_df  = data.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(),
      "| Test clients:", test_df['client_hash_id'].nunique())
print("Clients appearing in BOTH train and test (must be 0):", len(overlap))
print("Test base rate:", round(test_df['label_declined'].mean(), 4))

Train rows: 109325 | Test rows: 41118
Train clients: 30 | Test clients: 14
Clients appearing in BOTH train and test (must be 0): 0
Test base rate: 0.2323


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# Recompute the Week-4 baseline score, on the SAME test split, so the comparison is fair.
def normalize(s):
    lo, hi = s.min(), s.max()
    if hi == lo:
        return s * 0
    return (s - lo) / (hi - lo)

volume_score   = normalize(np.log1p(test_df['avg_impressions']))
position_score = normalize(-test_df['avg_position'].fillna(test_df['avg_position'].max()))
low_ctr_score  = normalize(-test_df['ctr'].fillna(0))

test_df = test_df.copy()
test_df['baseline_score'] = 0.40 * volume_score + 0.35 * position_score + 0.25 * low_ctr_score

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

X_train, y_train = train_df[FEATURES], train_df['label_declined']
X_test,  y_test  = test_df[FEATURES],  test_df['label_declined']

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)

test_df['logreg_score'] = logreg.predict_proba(X_test)[:, 1]
test_df['rf_score']     = rf.predict_proba(X_test)[:, 1]

def precision_at_k(df, score_col, label_col, k):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

rows = []
base_rate = y_test.mean()
for name, col in [('baseline (rule)', 'baseline_score'),
                   ('logistic regression', 'logreg_score'),
                   ('random forest', 'rf_score')]:
    rows.append({
        'method': name,
        'roc_auc': round(roc_auc_score(y_test, test_df[col]), 4),
        'avg_precision': round(average_precision_score(y_test, test_df[col]), 4),
        'precision@20': round(precision_at_k(test_df, col, 'label_declined', 20), 4),
        'precision@50': round(precision_at_k(test_df, col, 'label_declined', 50), 4),
    })

comparison = pd.DataFrame(rows)
print("Test-set base rate (random guessing baseline):", round(base_rate, 4))
comparison

Test-set base rate (random guessing baseline): 0.2323


,method,roc_auc,avg_precision,precision@20,precision@50
0,baseline (rule),0.7069,0.3235,0.10,0.10
1,logistic regression,0.6693,0.3362,0.20,0.22
2,random forest,0.7907,0.4427,0.35,0.40


**Record here after running (fill with the real table above):**
- Base rate: `0.2323`
- Baseline: ROC-AUC `0.7069`, avg precision `0.3235`, precision@20 `0.10`, precision@50 `0.10`
- Logistic regression: ROC-AUC `0.6693`, avg precision `0.3362`, precision@20 `0.20`, precision@50 `0.22`
- Random forest: ROC-AUC `0.7907`, avg precision `0.4427`, precision@20 `0.35`, precision@50 `0.40`
- Random Forest consistently surpassed both Logistic Regression and the baseline across all evaluation metrics, indicating its superior performance in this context.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# What does the logistic regression lean on? (coefficients, readable and sanity-checkable)
coef_table = pd.DataFrame({
    'feature': FEATURES,
    'logreg_coefficient': logreg.coef_[0].round(4)
}).sort_values('logreg_coefficient', key=abs, ascending=False)
print("Logistic regression coefficients (larger |value| = more influence):")
print(coef_table)

# What does the random forest lean on?
rf_importance = pd.DataFrame({
    'feature': FEATURES,
    'rf_importance': rf.feature_importances_.round(4)
}).sort_values('rf_importance', ascending=False)
print("\nRandom forest feature importances:")
print(rf_importance)

Logistic regression coefficients (larger |value| = more influence):
           feature  logreg_coefficient
3              ctr             -4.5293
4        days_seen              1.8745
1       avg_clicks             -0.4738
2     avg_position             -0.0073
0  avg_impressions              0.0037

Random forest feature importances:
           feature  rf_importance
0  avg_impressions         0.7309
3              ctr         0.1136
2     avg_position         0.0576
1       avg_clicks         0.0567
4        days_seen         0.0411


**Sanity check (fill after running):** is the top feature something that plausibly relates to decline (e.g. `ctr` or `avg_position`), or is it suspiciously perfect / hard to explain? A suspiciously dominant single feature is usually a sign of leakage, not skill — but this notebook only ever loaded feature-window columns as `FEATURES`, so there is no `tgt_*` column available to leak in the first place.

**Logistic Regression:**
- The most influential feature is `ctr` with a negative coefficient (-4.5293), meaning a higher CTR is associated with a lower probability of decline. This makes sense: good content generally has a high CTR and is less likely to decline.
- The second most influential feature is `days_seen` with a positive coefficient (1.8745), implying that content seen for more days is more likely to decline. This could reflect that older, more established content is more susceptible to eventual decline.

**Random Forest:**
- The top feature is `avg_impressions` (importance 0.7309), indicating that content with higher average impressions is a strong predictor in this model. This makes sense as impressions directly relate to content visibility and potential performance.
- The second most important feature is `ctr` (importance 0.1136), which, similar to logistic regression, is a plausible indicator of content quality.

In [6]:
# Three concrete wrong cases, from the stronger model's ranking (adjust rf_score/logreg_score below
# to whichever model wins in Section 3).
SCORE_COL = 'rf_score'  # <-- set to the winning model's score column once you have the table

false_positives = test_df[(test_df[SCORE_COL] > 0.5) & (test_df['label_declined'] == 0)] \
    .sort_values(SCORE_COL, ascending=False).head(3)
false_negatives = test_df[(test_df[SCORE_COL] < 0.3) & (test_df['label_declined'] == 1)] \
    .head(3)

print("Three false positives (model said 'declining', it wasn't):")
print(false_positives[['content_hash_id', 'avg_impressions', 'avg_position', 'ctr', SCORE_COL]])

print("\nThree false negatives (model missed a real decline):")
print(false_negatives[['content_hash_id', 'avg_impressions', 'avg_position', 'ctr', SCORE_COL]])

Three false positives (model said 'declining', it wasn't):
                content_hash_id  avg_impressions  avg_position       ctr  \
39804  content_39457d17e716086c      1716.500000     38.504867  0.000208   
38668  content_567d370cf1fdbd1d       829.785714     39.597704  0.000689   
38988  content_9a4594adab0f2c81       961.071429     34.168944  0.000669   

       rf_score  
39804  0.741590  
38668  0.731812  
38988  0.713211  

Three false negatives (model missed a real decline):
              content_hash_id  avg_impressions  avg_position       ctr  \
55   content_c8314a2cd6db9d47        39.714286     18.653051  0.003597   
104  content_22813e6b62a46d1b       205.500000      6.951690  0.002086   
113  content_7a3b22bca17537ed         8.857143      4.722635  0.008065   

     rf_score  
55   0.248532  
104  0.290611  
113  0.273302  


**Why these three false positives are hard (fill after running):** These false positives generally show high `avg_impressions` but very low `ctr` (e.g., `content_39457d17e716086c` with 1716.5 impressions and 0.0002 CTR). The model likely interprets the combination of high visibility and very low engagement as a strong signal of decline. However, since they are false positives, their actual target impressions did not drop below the 80% threshold. This could indicate cases where content has consistently high visibility but poor engagement, without experiencing a significant *decline*, or that the initial impression volume was so high that even with some drop, it didn't meet the decline criteria.

**Why these three false negatives are hard (fill after running):** For false negatives like `content_7a3b22bca17537ed` (avg_impressions = 8.857), the `avg_impressions` are very low. The model might struggle to confidently predict decline for pages with such low initial activity. When the baseline signal is already thin and noisy, a 20% drop in impressions might not create a sufficiently strong or clear signal for the model to classify it as a true decline, as it could be interpreted as natural fluctuation rather than a significant event.

**What this error pattern suggests about the label itself:** this month's `label_declined` is still the short within-month proxy named as a limitation back in ML-04 (a 14-day feature half vs 16-day target half) — some of these errors may reflect genuine model weakness, and some may reflect that the label itself is noisy on a window this short. I can't fully separate the two with one month of data; that's a limitation to carry into the capstone, not a claim that the model is worse than it is.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.